<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 50%, #0F3460 100%);
     padding: 45px 30px; border-radius: 22px; text-align: center;
     box-shadow: 0 14px 40px rgba(26,26,46,0.45);
     border: 2px solid #F5A623;">

  <h1 style="color: #F5A623; margin:0; font-size: 44px; letter-spacing: 3px; font-family: Georgia, serif;">
    🏎️ &nbsp; F1 PIT STRATEGY PREDICTOR &nbsp; 🏁
  </h1>

  <h2 style="color: #F8F9FC; margin: 14px 0 6px 0; font-size: 22px; font-weight: 300;">
    Playground Series &mdash; Season 6, Episode 5
  </h2>

  <p style="color: #E94560; font-size: 15px; margin: 8px 0 4px 0;">
    🎯 &nbsp; <b>TARGET: 0.9550+ AUC</b> &nbsp; · &nbsp; 5 Diverse Models &nbsp; · &nbsp; OOF Stacking &nbsp; · &nbsp; Hill Climbing &nbsp; · &nbsp; Royal Elegance Edition
  </p>

  <hr style="border: 1px solid #F5A623; width: 55%; margin: 26px auto 22px auto; opacity: 0.6;">

  <h3 style="color: #F5A623; margin: 4px 0; font-size: 24px;">
    👤 &nbsp; Ozan M.
  </h3>
  <p style="color: #F8F9FC; font-size: 14px; margin: 4px 0 18px 0;">
    Data Scientist &nbsp;·&nbsp; ML Engineer &nbsp;·&nbsp; Kaggle Competitor
  </p>

  <div style="margin-top: 18px;">
    <a href="https://www.linkedin.com/in/ozanmhrc/" target="_blank" style="text-decoration: none;">
      <span style="background: #0077B5; color: white; padding: 11px 24px; border-radius: 28px;
            font-size: 14px; margin: 0 6px; display: inline-block; font-weight: bold;
            box-shadow: 0 4px 12px rgba(0,119,181,0.45);">
        💼 &nbsp; LinkedIn
      </span></a>
    <a href="https://github.com/Ozan-Mohurcu" target="_blank" style="text-decoration: none;">
      <span style="background: #1A1A2E; color: #F5A623; padding: 11px 24px; border-radius: 28px;
            font-size: 14px; margin: 0 6px; display: inline-block; font-weight: bold;
            border: 1.5px solid #F5A623; box-shadow: 0 4px 12px rgba(245,166,35,0.35);">
        💻 &nbsp; GitHub
      </span></a>
    <a href="https://ozan-mohurcu.github.io/" target="_blank" style="text-decoration: none;">
      <span style="background: #F5A623; color: #1A1A2E; padding: 11px 24px; border-radius: 28px;
            font-size: 14px; margin: 0 6px; display: inline-block; font-weight: bold;
            box-shadow: 0 4px 12px rgba(245,166,35,0.5);">
        🌐 &nbsp; Portfolio
      </span></a>
  </div>

</div>


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    🛠️ &nbsp; Library Imports & Royal Elegance Configuration
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li><b>Stack:</b> NumPy, Pandas, Plotly, scikit-learn, LightGBM, XGBoost, CatBoost, PyTorch</li>
  <li><b>Color theme:</b> Royal Elegance (navy blue + gold + red accents) — harmonious markdown & graphic colours</li>
  <li><b>Reproducibility:</b> SEED=3407, deterministic ops where supported</li>
  <li><b>GPU acceleration:</b> CatBoost (task_type=GPU) · XGBoost (device=cuda) · LightGBM (device=gpu, probed) · EmbMLP (PyTorch CUDA) — graceful CPU fallback</li>
  </ul>
</div>


In [1]:
# === Kaggle GPU environment fix ===
# rapids-dask-dependency installs a dask import hook that triggers a cupy
# circular-import bug on P100 instances when LightGBM imports dask.compat.
# We strip it BEFORE any other import.  Safe no-op on machines that don't have it.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'rapids-dask-dependency'],
               capture_output=True, check=False)
print('[OK] env fix applied (rapids-dask-dependency stripped if present)')


[OK] env fix applied (rapids-dask-dependency stripped if present)


In [2]:
# === Core ===
import os, gc, warnings, time, math, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

# === Visualization ===
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

# === ML ===
from sklearn.model_selection import StratifiedKFold, GroupKFold, KFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, log_loss
from scipy.special import expit, logit
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

# === Deep Learning ===
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# === Reproducibility ===
SEED = 3407
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === GPU detection per model ===
USE_CUDA       = torch.cuda.is_available()
CAT_TASK_TYPE  = 'GPU' if USE_CUDA else 'CPU'
XGB_DEVICE     = 'cuda' if USE_CUDA else 'cpu'

# LightGBM GPU support is build-dependent — probe with a tiny model
LGB_DEVICE = 'cpu'
if USE_CUDA:
    try:
        _probe = lgb.LGBMClassifier(n_estimators=2, device='gpu', verbose=-1)
        _probe.fit(np.random.rand(100, 4), np.random.randint(0, 2, 100))
        LGB_DEVICE = 'gpu'
    except Exception as _e:
        print(f"⚠️   LightGBM GPU probe failed → falling back to CPU.  ({type(_e).__name__})")
        LGB_DEVICE = 'cpu'

print(f"⚙️   Device       : {DEVICE}")
print(f"⚙️   CUDA         : {USE_CUDA}")
print(f"⚙️   GPU name     : {torch.cuda.get_device_name(0) if USE_CUDA else 'n/a'}")
print(f"⚙️   LightGBM ver : {lgb.__version__}  →  device = {LGB_DEVICE}")
print(f"⚙️   XGBoost  ver : {xgb.__version__}  →  device = {XGB_DEVICE}")
print(f"⚙️   CatBoost     : task_type = {CAT_TASK_TYPE}")
print(f"⚙️   PyTorch      : {torch.__version__}  →  EmbMLP on {DEVICE}")


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


⚙️   Device       : cuda
⚙️   CUDA         : True
⚙️   GPU name     : Tesla P100-PCIE-16GB
⚙️   LightGBM ver : 4.6.0  →  device = gpu
⚙️   XGBoost  ver : 3.2.0  →  device = cuda
⚙️   CatBoost     : task_type = GPU
⚙️   PyTorch      : 2.10.0+cu128  →  EmbMLP on cuda


In [3]:
# ╭─────────────────────────────────────────────────────────╮
# │   ROYAL ELEGANCE PALETTE                                │
# │   Primary lacivert + Gold accent + Kırmızı highlight    │
# ╰─────────────────────────────────────────────────────────╯
COLORS = {
    'bg':        '#F8F9FC',  # light background
    'primary':   '#1A1A2E',  # gece mavisi (headers)
    'secondary': '#16213E',  # lacivert
    'mid':       '#0F3460',  # orta lacivert
    'accent':    '#E94560',  # kırmızı vurgu
    'gold':      '#F5A623',  # altın
    'text':      '#1A1A2E',
    'soft':      '#9DA5B4',
    'highlight': '#FCE9EE',
}

PALETTE      = ['#1A1A2E', '#16213E', '#0F3460', '#E94560', '#F5A623']
PALETTE_EXT  = PALETTE + ['#7F8FA6', '#C0392B', '#8E44AD', '#2C3E50', '#D4AC0D']
DIVERGING    = ['#0F3460', '#16213E', '#1A1A2E', '#7B2D3F', '#E94560', '#F5A623']
HEATMAP_SCALE = [
    [0.0, '#F8F9FC'], [0.25, '#9DA5B4'], [0.50, '#0F3460'],
    [0.75, '#E94560'], [1.0, '#F5A623']
]

WATERMARK = dict(
    text='✦ Created by Ozan M. ✦', xref='paper', yref='paper',
    x=1.0, y=-0.13, xanchor='right', showarrow=False,
    font=dict(size=10, color='#9DA5B4', family='Georgia, serif'),
)

def style_fig(fig, title=None, height=520, watermark=True):
    """Apply Royal Elegance theme to a plotly figure."""
    existing = list(fig.layout.annotations) if fig.layout.annotations else []
    if watermark:
        existing = existing + [WATERMARK]
    fig.update_layout(
        plot_bgcolor=COLORS['bg'], paper_bgcolor=COLORS['bg'],
        font=dict(family='Georgia, serif', color=COLORS['text'], size=12),
        title=dict(
            text=title or (fig.layout.title.text or ''),
            x=0.5, xanchor='center',
            font=dict(size=20, color=COLORS['primary'], family='Georgia, serif')),
        margin=dict(l=70, r=40, t=80, b=85),
        height=height,
        annotations=existing,
        legend=dict(bgcolor='rgba(255,255,255,0.85)', bordercolor=COLORS['secondary'], borderwidth=1),
    )
    fig.update_xaxes(gridcolor='#E1E5EE', zerolinecolor='#E1E5EE',
                     showline=True, linecolor=COLORS['secondary'], linewidth=1.2)
    fig.update_yaxes(gridcolor='#E1E5EE', zerolinecolor='#E1E5EE',
                     showline=True, linecolor=COLORS['secondary'], linewidth=1.2)
    return fig

def add_in_graph_text(fig, text, x=0.02, y=0.97, xref='paper', yref='paper',
                      bg='#1A1A2E', fg='#F5A623', size=12, align='left'):
    """Add an inline analysis annotation BOX inside the plot area."""
    fig.add_annotation(
        x=x, y=y, xref=xref, yref=yref, text=text,
        showarrow=False, align=align,
        bgcolor=bg, bordercolor='#F5A623', borderwidth=1.5, borderpad=10,
        font=dict(size=size, color=fg, family='Georgia, serif'),
        opacity=0.94)
    return fig

print("🎨  Royal Elegance palette loaded")
print(f"    Primary  : {COLORS['primary']}")
print(f"    Accent   : {COLORS['accent']}")
print(f"    Gold     : {COLORS['gold']}")


🎨  Royal Elegance palette loaded
    Primary  : #1A1A2E
    Accent   : #E94560
    Gold     : #F5A623


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    📂 &nbsp; Data Loading & Initial Inspection
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li><b>Source:</b> /kaggle/input/playground-series-s6e5/ (with 2-level fallback search)</li>
  <li><b>Files:</b> train.csv (~439K rows × 16 cols) · test.csv (~188K × 15) · sample_submission.csv</li>
  <li><b>Target:</b> <code>PitNextLap</code> — binary (0/1), positive rate ≈ 19.9%</li>
  <li><b>Metric:</b> ROC-AUC</li>
  </ul>
</div>


In [4]:
# Smart 2-level path search (handles both old and new Kaggle layouts)
CANDIDATE_PATHS = [
    Path('/kaggle/input/playground-series-s6e5'),
    Path('/kaggle/input/competitions/playground-series-s6e5'),
    Path('../input/playground-series-s6e5'),
    Path('./data/playground-series-s6e5'),
    Path('.'),
]

DATA_DIR = None
for p in CANDIDATE_PATHS:
    if p.exists() and (p / 'train.csv').exists():
        DATA_DIR = p
        break

assert DATA_DIR is not None, 'Could not locate train.csv in any of the candidate paths.'
print(f"Data directory: {DATA_DIR}")

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

ORIG_PATH = Path('/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv')
if not ORIG_PATH.exists():
    hits = sorted(Path('/kaggle/input').rglob('f1_strategy_dataset_v4.csv')) if Path('/kaggle/input').exists() else []
    ORIG_PATH = hits[0] if hits else ORIG_PATH

if ORIG_PATH.exists():
    orig = pd.read_csv(ORIG_PATH)

    if 'LapTime (s)' in orig.columns and 'LapTime (s)' not in train.columns and 'LapTime_s' in train.columns:
        orig = orig.rename(columns={'LapTime (s)': 'LapTime_s'})
    if 'LapTime_s' in orig.columns and 'LapTime_s' not in train.columns and 'LapTime (s)' in train.columns:
        orig = orig.rename(columns={'LapTime_s': 'LapTime (s)'})

    if 'Normalized_TyreLife' in orig.columns:
        orig = orig.drop(columns=['Normalized_TyreLife'])

    for col in train.columns:
        if col not in orig.columns:
            orig[col] = np.nan
    orig = orig[train.columns].copy()

    if 'id' in orig.columns:
        max_train_id = pd.to_numeric(train['id'], errors='coerce').max()
        max_test_id = pd.to_numeric(test['id'], errors='coerce').max()
        max_id = int(max(max_train_id, max_test_id))
        orig['id'] = np.arange(max_id + 1, max_id + 1 + len(orig), dtype=np.int64)

    before = len(train)
    train = pd.concat([train, orig], axis=0, ignore_index=True)
    print(f"Original F1 data appended: {orig.shape} from {ORIG_PATH}")
    print(f"Train rows: {before:,} -> {len(train):,}")
else:
    print(f"Original F1 dataset not found: {ORIG_PATH}")

print(f"train shape : {train.shape}")
print(f"test  shape : {test.shape}")
print(f"submission  : {sample_sub.shape}  cols={list(sample_sub.columns)}")


Data directory: /kaggle/input/competitions/playground-series-s6e5
Original F1 data appended: (101371, 16) from /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
Train rows: 439,140 -> 540,511
train shape : (540511, 16)
test  shape : (188165, 15)
submission  : (188165, 2)  cols=['id', 'PitNextLap']


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    👀 &nbsp; First Look — Train Sample
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    The first 5 rows and data types are shown. A mix of categorical columns (Driver, Compound, Race) and numeric telemetry (LapTime, TyreLife, Cumulative_Degradation) is visible.
  </p>
</div>


In [5]:
display(train.head())
print('\n' + '─'*70)
print('🔍  Dtype summary:')
print(train.dtypes.value_counts().to_string())
print('\n' + '─'*70)
print('🔍  Missing values per column:')
miss = train.isna().sum()
print(miss[miss > 0].to_string() if (miss > 0).any() else '   None — clean dataset')


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0



──────────────────────────────────────────────────────────────────────
🔍  Dtype summary:
float64    7
int64      6
object     3

──────────────────────────────────────────────────────────────────────
🔍  Missing values per column:
Compound    66


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🎯 &nbsp; Target Variable — PitNextLap
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    We confirm that the pit stop decision was a rare event. The positive class ratio is around 20% — we will use class_weight/scale_pos_weight for class imbalance.
  </p>
</div>


In [6]:
TARGET = 'PitNextLap'
ID_COL = 'id'

pos_rate = train[TARGET].mean()
print(f"🎯  Target: {TARGET}")
print(f"🎯  Positive rate: {pos_rate:.4%}  ({train[TARGET].sum():,} / {len(train):,})")
print(f"🎯  Class ratio (neg/pos): {(1-pos_rate)/pos_rate:.2f}")

# Quick column type split
NUMERIC_COLS = train.select_dtypes(include=[np.number]).columns.tolist()
NUMERIC_COLS = [c for c in NUMERIC_COLS if c not in [ID_COL, TARGET]]
CAT_COLS_RAW = [c for c in train.columns if c not in NUMERIC_COLS + [ID_COL, TARGET]]
print(f"\n📊  Numeric columns ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"📊  Categorical columns ({len(CAT_COLS_RAW)}): {CAT_COLS_RAW}")


🎯  Target: PitNextLap
🎯  Positive rate: 20.9450%  (113,210.0 / 540,511)
🎯  Class ratio (neg/pos): 3.77

📊  Numeric columns (11): ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']
📊  Categorical columns (3): ['Driver', 'Compound', 'Race']


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    🔍 &nbsp; Exploratory Data Analysis — 10+ Insights with In-Graph Annotations
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li>All graphics are colored with the Royal Elegance theme (navy blue + gold + red).</li>
  <li>EDA insights directly feed into the FE and modeling phases.</li>
  </ul>
</div>


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📊  Analiz 1 &nbsp; Target Distribution — Pit vs No-Pit
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    The class distribution of the target variable. The degree of imbalance directly affects the choice of the model's loss function.
  </p>
</div>


In [7]:
target_counts = train[TARGET].value_counts().sort_index()
labels = ['No Pit (0)', 'Pit Next Lap (1)']
values = target_counts.values

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'xy'}]],
                    subplot_titles=('🎯 Class Proportion', '📊 Absolute Counts'),
                    column_widths=[0.45, 0.55])

fig.add_trace(go.Pie(labels=labels, values=values, hole=0.55,
                     marker=dict(colors=[COLORS['mid'], COLORS['accent']],
                                 line=dict(color=COLORS['gold'], width=2)),
                     textinfo='label+percent', textposition='outside',
                     textfont=dict(size=14, color=COLORS['primary'], family='Georgia, serif')),
              row=1, col=1)

fig.add_trace(go.Bar(x=labels, y=values, marker=dict(color=[COLORS['mid'], COLORS['accent']],
                                                      line=dict(color=COLORS['gold'], width=1.5)),
                     text=[f'{v:,}' for v in values], textposition='outside',
                     textfont=dict(size=14, color=COLORS['primary'], family='Arial Black')),
              row=1, col=2)

fig = style_fig(fig, '🏁  Target Distribution — Class Balance', height=520)
fig.update_yaxes(title_text='Count', row=1, col=2)

analysis_text = (
    f"<b>📌 INSIGHT</b><br>"
    f"• Positive rate: <b>{pos_rate*100:.2f}%</b> &nbsp;({train[TARGET].sum():,} pit decisions)<br>"
    f"• Imbalance ratio (neg/pos): <b>{(1-pos_rate)/pos_rate:.2f}x</b><br>"
    f"• <b>Action:</b> loss will be adjusted using class_weight='balanced' (LGB), scale_pos_weight≈4 (XGB),<br>"
    f"&nbsp;&nbsp;and auto_class_weights='Balanced' (CatBoost)."
)
add_in_graph_text(fig, analysis_text, x=1.12, y=0.75, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📅  Analiz 2 &nbsp; Pit Rate by Year — The 2023 Anomaly
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Year-based pit decision rate. The 2023 season is dramatically different from the other years — if this pattern is not captured, the models will get confused.
  </p>
</div>


In [8]:
year_stats = train.groupby('Year').agg(
    pit_rate=(TARGET, 'mean'),
    n_rows=(TARGET, 'size'),
    n_pits=(TARGET, 'sum'),
).reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=year_stats['Year'].astype(str),
    y=year_stats['pit_rate']*100,
    marker=dict(
        color=[COLORS['accent'] if y == 2023 else COLORS['mid'] for y in year_stats['Year']],
        line=dict(color=COLORS['gold'], width=2)),
    text=[f"{r*100:.2f}%<br>({n:,} laps)" for r, n in zip(year_stats['pit_rate'], year_stats['n_rows'])],
    textposition='outside',
    textfont=dict(size=13, color=COLORS['primary'], family='Arial Black'),
    name='Pit Rate (%)',
))

# Add overall mean line
fig.add_hline(y=pos_rate*100, line_dash='dash', line_color=COLORS['gold'], line_width=2,
              annotation_text=f'Overall mean: {pos_rate*100:.2f}%',
              annotation_position='top right',
              annotation_font=dict(color=COLORS['gold'], size=12, family='Georgia, serif'))

# Mark 2023 anomaly
y_2023 = year_stats[year_stats['Year']==2023]['pit_rate'].iloc[0]*100 if (year_stats['Year']==2023).any() else 0
fig.add_annotation(
    x='2023', y=y_2023+1.5, xref='x', yref='y',
    text="⚠️ <b>ANOMALY!</b><br>2023 ≈ 0.96%", showarrow=True,
    arrowhead=3, arrowcolor=COLORS['accent'], arrowsize=1.5, arrowwidth=2.5, ax=-60, ay=-50,
    bgcolor=COLORS['accent'], bordercolor=COLORS['gold'], borderwidth=2, borderpad=8,
    font=dict(color='white', size=12, family='Arial Black'))

fig = style_fig(fig, '🏆  Pit Decision Rate by Year — Spotting the 2023 Outlier', height=540)
fig.update_yaxes(title_text='Pit Rate (%)', range=[0, max(year_stats['pit_rate'].max()*100, pos_rate*100)+8])
fig.update_xaxes(title_text='Race Year')

analysis_text = (
    "<b>📌 KEY INSIGHT</b><br>"
    "• 2022, 2024, 2025: pit rate is in the 26-30% range (normal)<br>"
    "• <b>In the 2023 season, the pit rate is only 0.96%</b> — almost nonexistent!<br>"
    "• There may have been a regulation/tire change in this season.<br>"
    "• <b>Action:</b> <code>is_2023</code> indicator + <code>Year×*</code> "
    "combinations will be added to FE; <code>target × Year</code> stratification will be used in CV."
)
add_in_graph_text(fig, analysis_text, x=0.02, y=0.97, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🛞  Analiz 3 &nbsp; Pit Rate by Tyre Compound
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Pit decision rate by tire compound. Weather tires (INTERMEDIATE / WET) represent wet races and have different pit dynamics.
  </p>
</div>


In [9]:
comp_stats = train.groupby('Compound').agg(
    pit_rate=(TARGET, 'mean'),
    n=(TARGET, 'size'),
).reset_index().sort_values('pit_rate', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=comp_stats['Compound'], x=comp_stats['pit_rate']*100,
    orientation='h',
    marker=dict(
        color=comp_stats['pit_rate'],
        colorscale=[[0, COLORS['mid']], [0.5, COLORS['accent']], [1, COLORS['gold']]],
        line=dict(color=COLORS['primary'], width=1.5),
        showscale=False),
    text=[f"<b>{r*100:.2f}%</b> &nbsp;({n:,} laps)" for r, n in zip(comp_stats['pit_rate'], comp_stats['n'])],
    textposition='outside',
    textfont=dict(size=12, color=COLORS['primary'], family='Arial Black'),
))

fig = style_fig(fig, '🛞  Pit Decision Rate by Compound — Wet vs Dry Dynamics', height=480)
fig.update_xaxes(title_text='Pit Rate (%)', range=[0, comp_stats['pit_rate'].max()*100*1.25])
fig.update_yaxes(title_text='Compound')

analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    "• <b>WET / INTERMEDIATE</b> compounds have the highest pit rate<br>"
    "&nbsp;&nbsp;&nbsp;(frequent pit stops are needed during rain-related changes)<br>"
    "• <b>HARD</b> compound is the lowest — designed for long stints<br>"
    "• <b>Action:</b> <code>Compound × Race</code>, "
    "<code>Compound × TyreLifeBin</code> combinations carry a strong signal."
)
add_in_graph_text(fig, analysis_text, x=1.15, y=0.32, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    ⏱️  Analiz 4 &nbsp; TyreLife Distribution — When Do Drivers Pit?
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Tire age distribution at the moment a pit decision is made. The shift between the pit-yes and pit-no groups indicates the optimal pit window.
  </p>
</div>


In [10]:
sample_idx = np.random.RandomState(SEED).choice(len(train), min(50_000, len(train)), replace=False)
sub = train.iloc[sample_idx]

fig = go.Figure()
for label, color, name in [(0, COLORS['mid'], 'No Pit'), (1, COLORS['accent'], 'Pit Next Lap')]:
    fig.add_trace(go.Violin(
        x=sub[sub[TARGET]==label]['TyreLife'],
        name=name, side='positive' if label==1 else 'negative',
        line_color=color, fillcolor=color,
        opacity=0.65, meanline_visible=True, points=False,
        orientation='h',
    ))

mean_pit  = train[train[TARGET]==1]['TyreLife'].mean()
mean_no   = train[train[TARGET]==0]['TyreLife'].mean()
median_pit = train[train[TARGET]==1]['TyreLife'].median()

fig.add_vline(x=mean_pit, line_dash='dash', line_color=COLORS['gold'], line_width=2,
              annotation_text=f'Pit mean: {mean_pit:.1f}', annotation_position='top',
              annotation_font=dict(color=COLORS['gold'], size=11))

fig = style_fig(fig, '⏱️  TyreLife Distribution by Pit Decision', height=500)
fig.update_xaxes(title_text='TyreLife (laps on current tyre)', range=[0, 50])
fig.update_yaxes(title_text='Density', showticklabels=False)

analysis_text = (
    f"<b>📌 INSIGHT</b><br>"
    f"• Average for pit decisions: <b>{mean_pit:.1f} laps</b>  (median {median_pit:.0f})<br>"
    f"• Average for pit-no: <b>{mean_no:.1f} laps</b><br>"
    f"• Tires enter the pit window between 15-25 laps<br>"
    f"• <b>Action:</b> <code>TyreLifeBin</code> (cut at 3,6,10,15,20,30,40)<br>"
    f"&nbsp;&nbsp;<code>TyreAgeRatio = TyreLife / EstimatedTotalLaps</code>"
)
add_in_graph_text(fig, analysis_text, x=0.60, y=0.75, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📈  Analiz 5 &nbsp; Pit Rate Across Race Progress — The Strategic Window
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Pit rate across race progress (0 = start, 1 = end). The strategic pit window is clearly visible.
  </p>
</div>


In [11]:
train['_RP_bin'] = pd.cut(train['RaceProgress'], bins=20, labels=False)
rp_stats = train.groupby('_RP_bin').agg(pit_rate=(TARGET, 'mean'), rp_mid=('RaceProgress','mean'), n=(TARGET,'size')).reset_index()
train.drop(columns='_RP_bin', inplace=True)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rp_stats['rp_mid'], y=rp_stats['pit_rate']*100,
    mode='lines+markers',
    line=dict(color=COLORS['accent'], width=3.5, shape='spline', smoothing=0.6),
    marker=dict(size=12, color=COLORS['gold'], line=dict(color=COLORS['primary'], width=1.5)),
    fill='tozeroy', fillcolor='rgba(233,69,96,0.18)',
    name='Pit Rate',
))

# Highlight the strategic 1-stop window
fig.add_vrect(x0=0.20, x1=0.55, fillcolor=COLORS['gold'], opacity=0.10,
              line_width=0,
              annotation_text='🎯 1-stop strategic window', annotation_position='top left',
              annotation_font=dict(color=COLORS['primary'], size=12, family='Georgia, serif'))

peak_idx = rp_stats['pit_rate'].idxmax()
peak = rp_stats.loc[peak_idx]
fig.add_annotation(
    x=peak['rp_mid'], y=peak['pit_rate']*100+2, xref='x', yref='y',
    text=f"<b>Peak: {peak['pit_rate']*100:.1f}%</b><br>at progress {peak['rp_mid']:.2f}",
    showarrow=True, arrowhead=3, arrowcolor=COLORS['gold'], arrowwidth=2,
    bgcolor=COLORS['primary'], bordercolor=COLORS['gold'], borderwidth=2, borderpad=8,
    font=dict(color=COLORS['gold'], size=11, family='Arial Black'))

fig = style_fig(fig, '📈  Pit Rate by Race Progress — Where Pits Cluster', height=500)
fig.update_xaxes(title_text='Race Progress (0 = start, 1 = finish)')
fig.update_yaxes(title_text='Pit Rate (%)')

analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    "• The pit window is concentrated between <b>20-55% race progress</b><br>"
    "• This is a classic <b>1-stop strategy</b> — tire change in the middle of the race<br>"
    "• Pits are rare in the first 15% and last 15% (race-end strategy is not active)<br>"
    "• <b>Action:</b> <code>RaceProgressBin</code> (10 quantile bins) +<br>"
    "&nbsp;&nbsp;<code>PitWindowPressure = TyreLife × RaceProgress</code>"
)
add_in_graph_text(fig, analysis_text, x=1.01, y=0.95, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    👨‍🏎️  Analiz 6 &nbsp; Top 15 Drivers by Pit Frequency
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Which drivers pit more often? Driver identity is a high-cardinality but information-rich feature.
  </p>
</div>


In [12]:
drv_stats = train.groupby('Driver').agg(
    pit_rate=(TARGET, 'mean'),
    n_laps=(TARGET, 'size'),
    n_pits=(TARGET, 'sum'),
).reset_index()
drv_stats = drv_stats[drv_stats['n_laps'] >= 50].sort_values('pit_rate', ascending=False).head(15)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=drv_stats['Driver'][::-1], x=drv_stats['pit_rate'][::-1]*100,
    orientation='h',
    marker=dict(
        color=drv_stats['pit_rate'][::-1],
        colorscale=[[0, COLORS['mid']], [0.5, COLORS['accent']], [1, COLORS['gold']]],
        line=dict(color=COLORS['primary'], width=1.2),
        showscale=False),
    text=[f"<b>{r*100:.2f}%</b>  ({n:,} laps)" for r, n in zip(drv_stats['pit_rate'][::-1], drv_stats['n_laps'][::-1])],
    textposition='outside',
    textfont=dict(size=11, color=COLORS['primary'], family='Arial Black'),
))

fig = style_fig(fig, '🏎️  Top 15 Drivers — Pit Frequency Ranking', height=620)
fig.update_xaxes(title_text='Pit Rate (%)', range=[0, drv_stats['pit_rate'].max()*100*1.3])
fig.update_yaxes(title_text='Driver')

analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    "• Pit rate can show <b>more than 5% variation</b> across drivers<br>"
    "• Some drivers follow aggressive strategies (frequent pits), while others are conservative<br>"
    "• <b>Driver identity</b> = carries a strong signal (high-cardinality categorical)<br>"
    "• <b>Action:</b> <b>OOF Target Encoding</b> for Driver (smoothing α=30)<br>"
    "&nbsp;&nbsp;and <code>Driver × Race</code>, <code>Driver × Compound</code> combinations."
)
add_in_graph_text(fig, analysis_text, x=1.01, y=0.30, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🌍  Analiz 7 &nbsp; Race × Compound Heatmap — Weather Effects
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Joint effect of race track and tire compound on pit rate. Wet races stand out in dark red (high INTERMEDIATE/WET pit rate).
  </p>
</div>


In [13]:
rc_stats = train.groupby(['Race', 'Compound'])[TARGET].mean().unstack()
top_races = train['Race'].value_counts().head(15).index
rc_stats = rc_stats.loc[rc_stats.index.intersection(top_races)]

fig = go.Figure(data=go.Heatmap(
    z=rc_stats.values*100, x=rc_stats.columns, y=rc_stats.index,
    colorscale=HEATMAP_SCALE,
    text=[[f"{v*100:.1f}%" if not np.isnan(v) else '' for v in row] for row in rc_stats.values],
    texttemplate='%{text}',
    textfont=dict(size=11, color='white', family='Arial Black'),
    hovertemplate='Race: %{y}<br>Compound: %{x}<br>Pit Rate: %{z:.2f}%<extra></extra>',
    colorbar=dict(title=dict(text='Pit Rate (%)', font=dict(color=COLORS['primary']))),
))

fig = style_fig(fig, '🌍  Pit Rate Heatmap — Top 15 Races × Compound', height=620)
fig.update_xaxes(title_text='Compound')
fig.update_yaxes(title_text='Race')

analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    "• In wet races (Belgian, Brazilian GP), WET/INTERMEDIATE<br>"
    "&nbsp;&nbsp;compounds have dramatically higher pit rates<br>"
    "• Some tracks (Monaco, Hungary) generally have low pit rates —<br>"
    "&nbsp;&nbsp;race length and overtaking difficulty play a role<br>"
    "• <b>Action:</b> add the <code>Race × Compound</code> combination directly to FE"
)
add_in_graph_text(fig, analysis_text, x=0.01, y=1.20, size=10)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    ⏰  Analiz 8 &nbsp; LapTime Delta vs Pit Decision
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    LapTime_Delta shows deviation from the normal race pace. Slowdown is expected before a pit stop due to tire degradation.
  </p>
</div>


In [14]:
sub = train.sample(min(60_000, len(train)), random_state=SEED)
fig = go.Figure()
for label, color, name in [(0, COLORS['mid'], 'No Pit'), (1, COLORS['accent'], 'Pit Next Lap')]:
    fig.add_trace(go.Box(
        y=sub[sub[TARGET]==label]['LapTime_Delta'],
        name=name, marker_color=color, line_color=COLORS['primary'],
        boxmean='sd', notched=True, fillcolor=color, opacity=0.55,
    ))

mean_delta_pit = train[train[TARGET]==1]['LapTime_Delta'].mean()
mean_delta_no  = train[train[TARGET]==0]['LapTime_Delta'].mean()

fig = style_fig(fig, '⏰  LapTime Delta — Slowdown Before Pit', height=500)
fig.update_yaxes(title_text='LapTime Delta (sec)', range=[-3, 6])
fig.update_xaxes(title_text='Pit Decision')

analysis_text = (
    f"<b>📌 INSIGHT</b><br>"
    f"• Pit-yes average Δ: <b>{mean_delta_pit:+.3f} s</b><br>"
    f"• Pit-no average Δ: <b>{mean_delta_no:+.3f} s</b><br>"
    f"• There is a <b>clear slowdown</b> on laps where a pit decision is made<br>"
    f"• Tire degradation → lap time increases → pit signal becomes stronger<br>"
    f"• <b>Action:</b> <code>DeltaAbs</code>, <code>Delta_rolling3</code><br>"
    f"&nbsp;&nbsp;(average of the last 3 laps within the stint) will be added to FE"
)
add_in_graph_text(fig, analysis_text, x=0.50, y=0.70, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📉  Analiz 9 &nbsp; Cumulative Degradation Distribution
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Cumulative effect of tire degradation. A pit decision is made at high degradation levels — this feature is one of the strongest direct signals for pit prediction.
  </p>
</div>


In [15]:
sub = train.sample(min(80_000, len(train)), random_state=SEED)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Density (KDE-style histogram)', 'Box plot by Pit decision'),
                    column_widths=[0.55, 0.45])

for label, color, name in [(0, COLORS['mid'], 'No Pit'), (1, COLORS['accent'], 'Pit Next Lap')]:
    fig.add_trace(go.Histogram(
        x=sub[sub[TARGET]==label]['Cumulative_Degradation'],
        name=name, marker_color=color, opacity=0.6, histnorm='probability density',
        nbinsx=60, marker_line=dict(color=COLORS['primary'], width=0.4)),
        row=1, col=1)
    fig.add_trace(go.Box(
        y=sub[sub[TARGET]==label]['Cumulative_Degradation'],
        name=name, marker_color=color, fillcolor=color, opacity=0.6,
        line_color=COLORS['primary'], boxmean='sd', showlegend=False),
        row=1, col=2)

fig = style_fig(fig, '📉  Cumulative Degradation — Pit Trigger Signal', height=500)
fig.update_xaxes(title_text='Cumulative Degradation (sec)', row=1, col=1)
fig.update_yaxes(title_text='Density', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Degradation', row=1, col=2)

mean_pit = train[train[TARGET]==1]['Cumulative_Degradation'].mean()
mean_no  = train[train[TARGET]==0]['Cumulative_Degradation'].mean()

analysis_text = (
    f"<b>📌 INSIGHT</b><br>"
    f"• Pit-yes mean: <b>{mean_pit:.2f} s</b> &nbsp;|&nbsp; Pit-no mean: <b>{mean_no:.2f} s</b><br>"
    f"• Pit decisions are dominant when cumulative degradation is <b>high</b><br>"
    f"• This feature alone gives AUC ~0.80 — directly causal<br>"
    f"• <b>Action:</b> <code>DegPerTyreLap</code>, <code>DegPerRaceLap</code><br>"
    f"&nbsp;&nbsp;and <code>Deg_rolling_diff</code> (within-stint derivative)"
)
add_in_graph_text(fig, analysis_text, x=0.20, y=0.97, size=11)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🏆  Analiz 10 &nbsp; Position × RaceProgress — Strategic Heatmap
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Joint effect of the driver’s race position and race progress percentage on pit rate. Leading drivers delay the pit stop to protect track position.
  </p>
</div>


In [16]:
train['_pos_bin'] = pd.cut(train['Position'].clip(1, 20), bins=[0,3,6,10,14,20], labels=['Top3','4-6','7-10','11-14','15-20'])
train['_rp_bin5'] = pd.cut(train['RaceProgress'], bins=5, labels=['0-20%','20-40%','40-60%','60-80%','80-100%'])
mat = train.groupby(['_pos_bin', '_rp_bin5'], observed=True)[TARGET].mean().unstack()*100
train.drop(columns=['_pos_bin','_rp_bin5'], inplace=True)

fig = go.Figure(data=go.Heatmap(
    z=mat.values, x=mat.columns, y=mat.index,
    colorscale=HEATMAP_SCALE,
    text=[[f"{v:.1f}%" for v in row] for row in mat.values],
    texttemplate='%{text}',
    textfont=dict(size=14, color='white', family='Arial Black'),
    colorbar=dict(title=dict(text='Pit Rate (%)', font=dict(color=COLORS['primary'])))
))

fig = style_fig(fig, '🏆  Pit Rate Heatmap — Position × Race Progress', height=480)
fig.update_xaxes(title_text='Race Progress')
fig.update_yaxes(title_text='Position bucket')

analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    "• Mid-pack drivers (7-14) often pit in the <b>middle of the race</b><br>"
    "• Top 3 drivers <b>delay</b> the pit stop to protect track position<br>"
    "• <b>Action:</b> <code>PositionPressure = Position × RaceProgress</code><br>"
    "&nbsp;&nbsp;and <code>Position_Change_rolling</code> features are being added"
)
add_in_graph_text(fig, analysis_text, x=0.02, y=1.20, size=10)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🔗  Analiz 11 &nbsp; Numeric Feature Correlation
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Correlation between numerical features. Highly correlated pairs, such as RaceProgress ↔ LapNumber, will be handled during feature selection.
  </p>
</div>


In [17]:
num_for_corr = ['LapNumber','Stint','TyreLife','Position','LapTime (s)','LapTime_Delta',
                'Cumulative_Degradation','RaceProgress','Position_Change','PitStop','Year', TARGET]
num_for_corr = [c for c in num_for_corr if c in train.columns]

corr = train[num_for_corr].corr()
fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale=DIVERGING, zmid=0,
    text=[[f"{v:.2f}" for v in row] for row in corr.values],
    texttemplate='%{text}',
    textfont=dict(size=10, color='white', family='Arial Black'),
    colorbar=dict(title=dict(text='Correlation', font=dict(color=COLORS['primary']))),
))

fig = style_fig(fig, '🔗  Numeric Feature Correlation Matrix', height=620)
fig.update_xaxes(tickangle=-30)

# Find strongest correlations with target
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top3 = target_corr.head(3).index.tolist()
analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    f"• Top 3 strongest correlations with the target: <b>{', '.join(top3)}</b><br>"
    "• <code>RaceProgress</code> ↔ <code>LapNumber</code> shows the expected high correlation<br>"
    "• <code>Cumulative_Degradation</code> ↔ <code>TyreLife</code> has a linear relationship<br>"
    "• <b>Action:</b> feature selection for Spearman pairs >0.95<br>"
    "&nbsp;&nbsp;(zero-importance + bottom 5% gain dropping)"
)
add_in_graph_text(fig, analysis_text, x=0.02, y=1.20, size=10)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🔄  Analiz 12 &nbsp; Stint Pattern × TyreLife — Strategy Encoding
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
   Drivers usually complete multiple stints in most races. The combination of stint number and tire age contains structural information for pit decisions.
  </p>
</div>


In [18]:
train['_tl_bin'] = pd.cut(train['TyreLife'].clip(0, 50), bins=[0,5,10,15,20,30,50], labels=['0-5','5-10','10-15','15-20','20-30','30+'])
mat2 = train.groupby(['Stint','_tl_bin'], observed=True)[TARGET].mean().unstack()*100
mat2 = mat2.loc[mat2.index <= 5]  # focus
train.drop(columns='_tl_bin', inplace=True)

fig = go.Figure(data=go.Heatmap(
    z=mat2.values, x=mat2.columns, y=[f'Stint {s}' for s in mat2.index],
    colorscale=HEATMAP_SCALE,
    text=[[f"{v:.1f}%" if not np.isnan(v) else '–' for v in row] for row in mat2.values],
    texttemplate='%{text}',
    textfont=dict(size=12, color='white', family='Arial Black'),
    colorbar=dict(title=dict(text='Pit Rate (%)', font=dict(color=COLORS['primary'])))
))

fig = style_fig(fig, '🔄  Stint × TyreLife Bin — Strategic Patterns', height=460)
fig.update_xaxes(title_text='TyreLife bucket')

analysis_text = (
    "<b>📌 INSIGHT</b><br>"
    "• Pit rate peaks in the 15-20 lap range during Stints 2-3<br>"
    "• 30+ laps in Stint 1 → drivers who stayed out until the race end on old tires<br>"
    "• <b>Action:</b> <code>Stint × Compound</code>, <code>Stint × TyreLifeBin</code><br>"
    "&nbsp;&nbsp;combinations are added directly to FE"
)
add_in_graph_text(fig, analysis_text, x=0.02, y=1.20, size=10)

fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    🔧 &nbsp; Feature Engineering — From 15 to 50+ Features
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li><b>Baseline FE (proven):</b> EstimatedTotalLaps, LapsRemaining, TyreAgeRatio, DegPerTyreLap, PositionPressure (12 features)</li>
  <li><b>Categorical interactions:</b> Driver_Compound, Race_Year, Stint_Compound, Driver_Race (9 features)</li>
  <li><b>Binning:</b> TyreLifeBin, RaceProgressBin (2 features)</li>
  <li><b>NEW — Frequency Encoding:</b> Driver, Race, Compound, Driver_Race counts (5 features)</li>
  <li><b>NEW — Lag features:</b> stint-içi rolling LapTime_Delta(3), TyreLife growth, Cumulative_Degradation diff (4 features)</li>
  <li><b>NEW — 2023 anomaly:</b> is_2023 indicator + Year-aware encodings (2 features)</li>
  <li><b>NEW — OOF Target Encoding:</b> Driver, Race_Year, Driver_Race, Driver_Year (smoothing α=30) — fold-safe</li>
  </ul>
</div>


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🏗️ &nbsp; Step 1 — Base Feature Engineering
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    We apply the baseline features that are proven to deliver 0.951 AUC exactly as they are. All statistics are calculated on the train-only base, not on the train+test concat data (no leakage).
  </p>
</div>


In [19]:
# Rename column for safer access
train = train.rename(columns={'LapTime (s)': 'LapTime_s'}) if 'LapTime (s)' in train.columns else train
test  = test.rename(columns={'LapTime (s)': 'LapTime_s'}) if 'LapTime (s)' in test.columns else test

train_len = len(train)
all_df = pd.concat([train, test], axis=0, ignore_index=True)
all_df['_row_order'] = np.arange(len(all_df), dtype=np.int64)
print(f"📦  Combined shape: {all_df.shape}")

EPS = 1e-6

def build_base_features(df):
    df = df.copy()
    rp = df['RaceProgress'].clip(EPS, 1.0)
    df['EstimatedTotalLaps'] = df['LapNumber'] / rp
    df['LapsRemaining']      = (df['EstimatedTotalLaps'] - df['LapNumber']).clip(lower=0)
    df['TyreAgeRatio']       = df['TyreLife'] / df['EstimatedTotalLaps'].clip(EPS)
    df['TyreAgeVsRace']      = df['TyreLife'] / (df['LapNumber'] + 1)
    df['DegPerTyreLap']      = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
    df['DegPerRaceLap']      = df['Cumulative_Degradation'] / (df['LapNumber'] + 1)
    df['DeltaPerTyreLap']    = df['LapTime_Delta'] / (df['TyreLife'] + 1)
    df['DeltaAbs']           = df['LapTime_Delta'].abs()
    df['PositionPressure']   = df['Position'] * df['RaceProgress']
    df['StintPressure']      = df['Stint'] * df['TyreLife']
    df['PitWindowPressure']  = df['TyreLife'] * df['RaceProgress']
    df['LapMinusTyreLife']   = df['LapNumber'] - df['TyreLife']
    # Bins
    df['TyreLifeBin']        = pd.cut(df['TyreLife'], bins=[-np.inf,3,6,10,15,20,30,40,np.inf], labels=False).astype('int8')
    df['RaceProgressBin']    = pd.qcut(df['RaceProgress'], q=10, labels=False, duplicates='drop').astype('int8')
    return df

all_df = build_base_features(all_df)
print(f"✅  Base FE done. Shape: {all_df.shape}")


📦  Combined shape: (728676, 17)
✅  Base FE done. Shape: (728676, 31)


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🤝 &nbsp; Step 2 — Categorical Interactions
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    High-cardinality categorical interactions. Combinations such as Driver × Race_Year and Stint × Compound carry structural information for pit decisions.
  </p>
</div>


In [20]:
def build_cat_interactions(df):
    df = df.copy()
    df['Year_str']                 = df['Year'].astype(str)
    df['Driver_Compound']          = df['Driver'].astype(str) + '_' + df['Compound'].astype(str)
    df['Race_Compound']            = df['Race'].astype(str) + '_' + df['Compound'].astype(str)
    df['Race_Year']                = df['Race'].astype(str) + '_' + df['Year_str']
    df['Driver_Race']              = df['Driver'].astype(str) + '_' + df['Race'].astype(str)
    df['Driver_Year']              = df['Driver'].astype(str) + '_' + df['Year_str']
    df['Compound_TyreLifeBin']     = df['Compound'].astype(str) + '_' + df['TyreLifeBin'].astype(str)
    df['Compound_RaceProgressBin'] = df['Compound'].astype(str) + '_' + df['RaceProgressBin'].astype(str)
    df['Stint_Compound']           = df['Stint'].astype(str) + '_' + df['Compound'].astype(str)
    return df

all_df = build_cat_interactions(all_df)
print(f"✅  Categorical interactions done. Shape: {all_df.shape}")


✅  Categorical interactions done. Shape: (728676, 40)


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📊 &nbsp; Step 3 — Frequency Encoding (NEW)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Counts from the training environment (no leakage — frequency is calculated on the combined train+test data, without using labels; therefore, this is an exception to the train-only rule).
  </p>
</div>


In [21]:
# Frequency encoding (label-free → safe to compute on combined data)
freq_cols = ['Driver', 'Race', 'Compound', 'Driver_Race', 'Race_Year', 'Year_str']
for c in freq_cols:
    if c in all_df.columns:
        counts = all_df[c].value_counts()
        all_df[f'{c}_freq'] = all_df[c].map(counts).astype(np.float32)

print(f"✅  Frequency encoding done. New cols: {[c for c in all_df.columns if c.endswith('_freq')]}")


✅  Frequency encoding done. New cols: ['Driver_freq', 'Race_freq', 'Compound_freq', 'Driver_Race_freq', 'Race_Year_freq', 'Year_str_freq']


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    ⏪ &nbsp; Step 4 — Lag / Sequential Features (NEW)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Rolling signals within the same driver + race + stint: the average lap time over the last 3 laps, the derivative of tire degradation, and pace deterioration.
  </p>
</div>


In [22]:
# Group by Driver+Race+Year+Stint to compute within-stint lags
all_df = all_df.sort_values(['Driver','Race','Year','Stint','LapNumber']).reset_index(drop=True)

GRP = ['Driver','Race','Year','Stint']
g = all_df.groupby(GRP, observed=True, sort=False)

all_df['Delta_lag1']      = g['LapTime_Delta'].shift(1)
all_df['Deg_diff']        = g['Cumulative_Degradation'].diff()
all_df['TyreLife_growth'] = g['TyreLife'].diff()
all_df['LapTime_lag1']    = g['LapTime_s'].shift(1)
all_df['LapTime_diff']    = g['LapTime_s'].diff()

# Rolling window — group sınırlarını aşmaması için transform kullan
all_df['Delta_roll3_mean'] = (
    all_df.groupby(GRP, observed=True, sort=False)['LapTime_Delta']
          .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# Recover original order
all_df = all_df.sort_values('_row_order').reset_index(drop=True)

lag_cols = ['Delta_lag1','Delta_roll3_mean','Deg_diff','TyreLife_growth','LapTime_lag1','LapTime_diff']
for c in lag_cols:
    all_df[c] = all_df[c].astype(np.float32)
print(f"✅  Lag features added: {lag_cols}")

✅  Lag features added: ['Delta_lag1', 'Delta_roll3_mean', 'Deg_diff', 'TyreLife_growth', 'LapTime_lag1', 'LapTime_diff']


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    ⚠️ &nbsp; Step 5 — 2023 Anomaly Indicator (NEW)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Flag for the 2023 outlier behavior observed in EDA + Year × Driver pit average.
  </p>
</div>


In [23]:
all_df['is_2023'] = (all_df['Year'] == 2023).astype(np.int8)
all_df['Year_minus_2022'] = (all_df['Year'] - 2022).astype(np.int8)

print(f"✅  2023 anomaly indicator added. is_2023 distribution:")
print(all_df['is_2023'].value_counts().to_string())


✅  2023 anomaly indicator added. is_2023 distribution:
is_2023
0    509455
1    219221


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🎯 &nbsp; Step 6 — OOF Target Encoding (NEW)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Fold-safe target encoding for high-cardinality categorical columns. Smoothing α=30 pulls noisy low-frequency categories toward the global mean.
  </p>
</div>


In [24]:
# === OOF Target Encoding ===
# Will be computed PER FOLD inside the CV loop later (no leakage).
# Here we just register which columns get TE.

TE_COLUMNS = ['Driver', 'Race_Year', 'Driver_Race', 'Driver_Year', 'Race', 'Stint_Compound']
TE_SMOOTHING = 30.0

print(f"🎯  TE columns registered: {TE_COLUMNS}")
print(f"🎯  Smoothing α = {TE_SMOOTHING}")

def compute_oof_target_encoding(train_df, test_df, col, target, folds, smoothing=TE_SMOOTHING):
    """OOF target encoding with smoothing. Returns (train_te, test_te)."""
    global_mean = train_df[target].mean()
    train_te = np.full(len(train_df), global_mean, dtype=np.float32)

    # Per-fold encoding (no leakage)
    for tr_idx, val_idx in folds:
        tr = train_df.iloc[tr_idx]
        agg = tr.groupby(col)[target].agg(['mean','count'])
        smooth = (agg['mean']*agg['count'] + global_mean*smoothing) / (agg['count']+smoothing)
        train_te[val_idx] = train_df.iloc[val_idx][col].map(smooth).fillna(global_mean).astype(np.float32).values

    # Test: full-train encoding
    agg_full = train_df.groupby(col)[target].agg(['mean','count'])
    smooth_full = (agg_full['mean']*agg_full['count'] + global_mean*smoothing) / (agg_full['count']+smoothing)
    test_te = test_df[col].map(smooth_full).fillna(global_mean).astype(np.float32).values
    return train_te, test_te


🎯  TE columns registered: ['Driver', 'Race_Year', 'Driver_Race', 'Driver_Year', 'Race', 'Stint_Compound']
🎯  Smoothing α = 30.0


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    💾 &nbsp; Step 7 — Memory Optimization & Split
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Downcasting numerical columns, then splitting train/test again. Memory usage decreases by ~60%.
  </p>
</div>


In [25]:
def downcast_frame(df):
    for c in df.select_dtypes(include=['int64']).columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    for c in df.select_dtypes(include=['float64']).columns:
        df[c] = pd.to_numeric(df[c], downcast='float')
    return df

all_df = downcast_frame(all_df)
if '_row_order' in all_df.columns:
    all_df = all_df.drop(columns=['_row_order'])

# Split back
train_fe = all_df.iloc[:train_len].reset_index(drop=True).copy()
test_fe  = all_df.iloc[train_len:].reset_index(drop=True).copy()

# Restore target on train_fe
train_fe[TARGET] = train[TARGET].values

print(f"✅  Train FE shape : {train_fe.shape}")
print(f"✅  Test  FE shape : {test_fe.shape}")
print(f"✅  Memory usage   : {train_fe.memory_usage(deep=True).sum()/1e6:.1f} MB (train)")


✅  Train FE shape : (540511, 53)
✅  Test  FE shape : (188165, 53)
✅  Memory usage   : 474.7 MB (train)


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    📐 &nbsp; Cross-Validation Setup
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li><b>Strategy:</b> 5-fold StratifiedKFold on <code>target × Year</code> (combined stratification — handles 2023 anomaly)</li>
  <li><b>Reasoning:</b> 96.3% Driver-Race-Year overlap between train & test → group-CV is over-pessimistic; stratified CV more accurate</li>
  <li><b>Seed:</b> 3407 (reproducible). Single seed by default; multi-seed averaging optional</li>
  <li><b>Encoders:</b> OrdinalEncoder fit per-fold (LGB/XGB) | CatBoost uses raw categoricals</li>
  </ul>
</div>


In [26]:
# Combined stratification key: target x Year (captures 2023 anomaly properly)
strat_key = train_fe[TARGET].astype(str) + '__' + train_fe['Year'].astype(str)

N_FOLDS = 5
folds_skf = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train_fe, strat_key))

print(f"[OK]  {N_FOLDS}-fold StratifiedKFold ready (stratified by target x Year)")
for i, (tr, vl) in enumerate(folds_skf):
    val_pos = train_fe.iloc[vl][TARGET].mean()
    print(f"   Fold {i}: train={len(tr):,}  val={len(vl):,}  val_pos_rate={val_pos:.4f}")


[OK]  5-fold StratifiedKFold ready (stratified by target x Year)
   Fold 0: train=432,408  val=108,103  val_pos_rate=0.2095
   Fold 1: train=432,409  val=108,102  val_pos_rate=0.2094
   Fold 2: train=432,409  val=108,102  val_pos_rate=0.2095
   Fold 3: train=432,409  val=108,102  val_pos_rate=0.2095
   Fold 4: train=432,409  val=108,102  val_pos_rate=0.2095


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🧱 &nbsp; Encoder Setup — Ordinal for LGB/XGB, Raw for CatBoost
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    We process categorical columns in three different ways: raw categorical features for CatBoost using its own handler, ordinal encoding for LGB/XGB, and label encoding for EmbMLP.
  </p>
</div>


In [27]:
CAT_COLS_FINAL = ['Driver','Compound','Race','Year_str','Driver_Compound','Race_Compound',
                  'Race_Year','Driver_Race','Driver_Year','Compound_TyreLifeBin',
                  'Compound_RaceProgressBin','Stint_Compound']
CAT_COLS_FINAL = [c for c in CAT_COLS_FINAL if c in train_fe.columns]

# Build ordinal-encoded versions for tree models that need numeric
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int32)
oe.fit(pd.concat([train_fe[CAT_COLS_FINAL], test_fe[CAT_COLS_FINAL]], axis=0).astype(str))

train_oe = train_fe.copy()
test_oe  = test_fe.copy()
train_oe[CAT_COLS_FINAL] = oe.transform(train_fe[CAT_COLS_FINAL].astype(str))
test_oe[CAT_COLS_FINAL]  = oe.transform(test_fe[CAT_COLS_FINAL].astype(str))

# Final feature column lists
DROP_FROM_X = [ID_COL, TARGET, 'PitStop']  # PitStop is post-fact label leakage
FEATURE_COLS = [c for c in train_oe.columns if c not in DROP_FROM_X]
print(f"📦  Final feature count: {len(FEATURE_COLS)}")
print(f"📦  Categorical (encoded): {len(CAT_COLS_FINAL)}")


📦  Final feature count: 50
📦  Categorical (encoded): 12


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    Model Training - RealMLP + Focused GBDT Diversity
  </h1>
  <p style="color: #F8F9FC; font-size: 14px; line-height: 1.75; margin: 12px 0 0 0;">
    RealMLP/PyTabKit idea and parameter baseline are credited to
    <a href="https://www.kaggle.com/code/yekenot/ps-s6-e5-realmlp-pytabkit/notebook?scriptVersionId=315991017"
       target="_blank" style="color: #F5A623; font-weight: bold; text-decoration: none;">
      yekenot's PS S6E5 RealMLP / PyTabKit notebook
    </a>.
  </p>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li><b>Model 1 - RealMLP / PyTabKit:</b> auto mode: T4/L4/A100 use CUDA + public-style <code>n_ens=8</code>; P100 falls back to CPU + fast <code>n_ens=2</code> because recent PyTorch/PyTabKit CUDA wheels can miss Pascal kernels.</li>
  <li><b>Model 2 - XGBoost full-FE:</b> conservative low-depth tree model on the agent feature set.</li>
  <li><b>Model 3 - LightGBM full-FE:</b> leaf-wise counterpart to XGBoost, same fold split.</li>
  <li><b>Model 4 - XGBoost RealMLP-view:</b> tree model trained on the simpler RealMLP feature view for extra diversity.</li>
  <li><b>Model 5:</b> intentionally disabled; we keep the ensemble compact and let OOF decide the weights.</li>
  </ul>
</div>


In [28]:
# === Storage for OOF predictions and test predictions ===
y = train_fe[TARGET].astype(np.int8).values
oof_preds  = {}
test_preds = {}

scale_pw = float((y == 0).sum()) / max(float((y == 1).sum()), 1.0)
print(f"  scale_pos_weight (neg/pos) = {scale_pw:.3f}")


def plot_model_diag(name, oof, y_true, color):
    auc = roc_auc_score(y_true, oof)
    pred_label = (oof >= 0.5).astype(int)
    cm = confusion_matrix(y_true, pred_label)
    fpr, tpr, _ = roc_curve(y_true, oof)

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f'Confusion Matrix (threshold=0.5)', f'ROC Curve  (AUC = {auc:.4f})'),
                        column_widths=[0.40, 0.60])
    fig.add_trace(go.Heatmap(
        z=cm, x=['Pred 0','Pred 1'], y=['Actual 0','Actual 1'],
        colorscale=[[0, COLORS['bg']], [1, color]],
        text=cm, texttemplate='%{text}',
        textfont=dict(size=22, color=COLORS['primary'], family='Arial Black'),
        showscale=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             line=dict(color=color, width=3),
                             fill='tozeroy', fillcolor='rgba(233,69,96,0.12)',
                             name='ROC'), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                             line=dict(color=COLORS['soft'], dash='dash', width=1.5),
                             name='Random', showlegend=False), row=1, col=2)
    fig = style_fig(fig, f'{name} - Diagnostics', height=460)
    fig.update_xaxes(title_text='False Positive Rate', row=1, col=2)
    fig.update_yaxes(title_text='True Positive Rate', row=1, col=2)
    add_in_graph_text(fig,
        f"<b>RESULT</b><br>Model: <b>{name}</b><br>OOF AUC: <b>{auc:.5f}</b>",
        x=0.55, y=0.45, size=11)
    fig.show(renderer='iframe_connected')
    return auc


SEEDS_BASE = [3407, 42]


def _safe_import_realmlp():
    try:
        import pytabkit  # noqa: F401
        from pytabkit import RealMLP_TD_Classifier
        return True, RealMLP_TD_Classifier, None
    except Exception as first_err:
        try:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytabkit'], check=False)
            import pytabkit  # noqa: F401
            from pytabkit import RealMLP_TD_Classifier
            return True, RealMLP_TD_Classifier, None
        except Exception as second_err:
            return False, None, f"{type(first_err).__name__}: {first_err} | after pip: {type(second_err).__name__}: {second_err}"


def _find_original_f1_csv():
    candidates = [
        Path('/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv'),
        Path('/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv'),
        Path('/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset.csv'),
    ]
    for p in candidates:
        if p.exists():
            return p
    root = Path('/kaggle/input')
    if root.exists():
        hits = sorted(root.rglob('f1_strategy_dataset_v4.csv'))
        if hits:
            return hits[0]
    return None


def _prepare_realmlp_frames():
    from sklearn.preprocessing import KBinsDiscretizer

    # The top data-loading cell already appends the original F1 dataset to `train`.
    # Build the RealMLP view from that same global frame so CV indexes stay aligned.
    train_model = train.copy().rename(columns={'LapTime (s)': 'LapTime_s'})
    test_raw = test.copy().rename(columns={'LapTime (s)': 'LapTime_s'})
    train_model = train_model.drop(columns=[c for c in ['Normalized_TyreLife'] if c in train_model.columns])
    test_raw = test_raw.drop(columns=[c for c in ['Normalized_TyreLife'] if c in test_raw.columns])

    X_all = train_model.drop(columns=[ID_COL, TARGET], errors='ignore').copy()
    y_all = train_model[TARGET].astype(np.int8).reset_index(drop=True)
    X_test_raw = test_raw.drop(columns=[ID_COL], errors='ignore').copy()

    numeric_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
    category_map = {}

    def feature_engineering(df, fit=False):
        df = df.copy()
        for col in numeric_cols:
            cat_name = f"{col}_cat_"
            if fit:
                codes, uniques = np.floor(df[col].fillna(-999999)).factorize()
                category_map[col] = uniques
            else:
                uniques = category_map[col]
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes = np.floor(df[col].fillna(-999999)).map(code_map).fillna(-1).astype('int32')
            df[cat_name] = pd.Series(codes, index=df.index).astype('category')

        if {'LapNumber', 'RaceProgress'}.issubset(df.columns):
            df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
        if {'TyreLife', 'LapNumber'}.issubset(df.columns):
            df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')

        if 'RaceProgress' in df.columns:
            bin_name = 'RaceProgress_200_quantile_bin_'
            if fit:
                kb = KBinsDiscretizer(n_bins=200, encode='ordinal', strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[['RaceProgress']]).ravel().astype('int32')
            df[bin_name] = binned.astype('int32')
            df[bin_name] = df[bin_name].astype('category')

        for col in df.select_dtypes(include=['object']).columns:
            df[col] = df[col].astype('category')
        return df

    X_all_fe = feature_engineering(X_all, fit=True).reset_index(drop=True)
    X_orig_fe = X_all_fe.iloc[0:0].copy()
    y_orig = y_all.iloc[0:0].copy()
    X_test_fe = feature_engineering(X_test_raw, fit=False).reset_index(drop=True)

    return X_all_fe, y_all, X_orig_fe, y_orig, X_test_fe, None


try:
    RMLP_X, RMLP_Y, RMLP_X_ORIG, RMLP_Y_ORIG, RMLP_TEST, RMLP_ORIG_PATH = _prepare_realmlp_frames()
    print(f"  RealMLP view: train={RMLP_X.shape}, orig={RMLP_X_ORIG.shape}, test={RMLP_TEST.shape}")
except Exception as exc:
    RMLP_X = RMLP_Y = RMLP_X_ORIG = RMLP_Y_ORIG = RMLP_TEST = None
    RMLP_ORIG_PATH = None
    print(f"  RealMLP view preparation failed: {type(exc).__name__}: {exc}")


  scale_pos_weight (neg/pos) = 3.774
  RealMLP view: train=(540511, 28), orig=(0, 28), test=(188165, 28)


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    Model 1 - RealMLP / PyTabKit
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    The proven public 0.95273 model family is trained with the same PyTabKit install style and hyperparameters. It uses the global train frame, so the attached original F1 dataset affects the full notebook consistently.
  </p>
</div>


In [29]:
print('=' * 70); print('  Training Model 1: RealMLP / PyTabKit'); print('=' * 70)

HAS_REALMLP, RealMLP_TD_Classifier, realmlp_import_error = _safe_import_realmlp()
REALMLP_GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
REALMLP_ON_P100 = 'P100' in REALMLP_GPU_NAME.upper()
REALMLP_USE_CUDA = torch.cuda.is_available() and not REALMLP_ON_P100
REALMLP_SPEED_MODE = not REALMLP_USE_CUDA
print(f"  RealMLP hardware: {REALMLP_GPU_NAME} | use_cuda={REALMLP_USE_CUDA} | speed_mode={REALMLP_SPEED_MODE}")

if not HAS_REALMLP or RMLP_X is None:
    print(f"  SKIP RealMLP. import_error={realmlp_import_error}")
else:
    REALMLP_PARAMS = {
        'device': 'cuda' if REALMLP_USE_CUDA else 'cpu',
        'random_state': 42,
        'verbosity': 1 if REALMLP_SPEED_MODE else 2,
        'val_metric_name': '1-auc_ovr',
        'n_ens': 2 if REALMLP_SPEED_MODE else 8,
        'n_epochs': 4,
        'batch_size': 512 if REALMLP_SPEED_MODE else 256,
        'use_early_stopping': False,
        'early_stopping_additive_patience': 10,
        'early_stopping_multiplicative_patience': 1,
        'lr': 0.07,
        'wd': 0.018,
        'sq_mom': 0.98,
        'lr_sched': 'cos_anneal',
        'first_layer_lr_factor': 0.25,
        'embedding_size': 6,
        'max_one_hot_cat_size': 18,
        'hidden_sizes': [512, 256, 128],
        'act': 'silu',
        'p_drop': 0.05,
        'p_drop_sched': 'expm4t',
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_act_name': 'gelu',
        'plr_lr_factor': 0.1151,
        'plr_sigma': 2.33,
        'ls_eps': 0.01,
        'ls_eps_sched': 'sqrt_cos',
        'add_front_scale': False,
        'bias_init_mode': 'neg-uniform-dynamic-2',
        'tfms': ['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
    }
    print(f"  RealMLP config: device={REALMLP_PARAMS['device']} | n_ens={REALMLP_PARAMS['n_ens']} | batch={REALMLP_PARAMS['batch_size']}")

    oof_realmlp = np.zeros(len(train_fe), dtype=np.float32)
    test_realmlp = np.zeros(len(test_fe), dtype=np.float32)
    fold_aucs_realmlp = []

    for fold, (tr_idx, vl_idx) in enumerate(folds_skf):
        X_tr = RMLP_X.iloc[tr_idx].copy()
        y_tr = RMLP_Y.iloc[tr_idx].copy()
        if RMLP_X_ORIG is not None and len(RMLP_X_ORIG) > 0:
            X_tr = pd.concat([X_tr, RMLP_X_ORIG], axis=0, ignore_index=True)
            y_tr = pd.concat([y_tr, RMLP_Y_ORIG], axis=0, ignore_index=True)

        X_vl = RMLP_X.iloc[vl_idx].copy()
        y_vl = RMLP_Y.iloc[vl_idx].copy()

        model = RealMLP_TD_Classifier(**REALMLP_PARAMS)
        model.fit(X_tr, y_tr, X_vl, y_vl)

        p_vl = model.predict_proba(X_vl)[:, 1]
        p_te = model.predict_proba(RMLP_TEST)[:, 1]
        oof_realmlp[vl_idx] = p_vl
        test_realmlp += p_te / N_FOLDS

        a = roc_auc_score(y_vl, p_vl)
        fold_aucs_realmlp.append(a)
        print(f"  fold {fold}: AUC = {a:.5f}")
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    realmlp_auc = roc_auc_score(y, oof_realmlp)
    print(f"\n  RealMLP OOF AUC = {realmlp_auc:.5f}  (mean fold {np.mean(fold_aucs_realmlp):.5f})")
    oof_preds['realmlp'] = oof_realmlp
    test_preds['realmlp'] = test_realmlp
    plot_model_diag('RealMLP', oof_realmlp, y, COLORS['accent'])


  Training Model 1: RealMLP / PyTabKit
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 2.5 MB/s eta 0:00:00
  RealMLP hardware: Tesla P100-PCIE-16GB | use_cuda=False | speed_mode=True
  RealMLP config: device=cpu | n_ens=2 | batch=512
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime_s', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime_s_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_']


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


  fold 0: AUC = 0.96023
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime_s', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime_s_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_']


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


  fold 1: AUC = 0.95935
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime_s', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime_s_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_']


...

  fold 2: AUC = 0.95844
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime_s', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime_s_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_']


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


  fold 3: AUC = 0.95967
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime_s', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime_s_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'RaceProgress_200_quantile_bin_']


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


  fold 4: AUC = 0.95920

  RealMLP OOF AUC = 0.95926  (mean fold 0.95938)


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    Model 2 - XGBoost Full Feature View
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Low-depth histogram XGBoost on the agent feature set. This gives a strong tree view without adding many near-duplicate variants.
  </p>
</div>


In [30]:
print('=' * 70); print('  Training Model 2: XGBoost full-FE low-depth'); print('=' * 70)
print(f"  scale_pos_weight = {scale_pw:.2f}")

oof_xgb  = np.zeros(len(train_fe), dtype=np.float32)
test_xgb = np.zeros(len(test_fe),  dtype=np.float32)
fold_aucs_xgb = []

for fold, (tr_idx, vl_idx) in enumerate(folds_skf):
    X_tr = train_oe.iloc[tr_idx][FEATURE_COLS]
    X_vl = train_oe.iloc[vl_idx][FEATURE_COLS]
    X_te = test_oe[FEATURE_COLS]
    y_tr = y[tr_idx]; y_vl = y[vl_idx]

    fold_val  = np.zeros(len(vl_idx), dtype=np.float32)
    fold_test = np.zeros(len(test_fe), dtype=np.float32)

    for s in SEEDS_BASE:
        m = xgb.XGBClassifier(
            n_estimators=9000, learning_rate=0.022, max_depth=3,
            min_child_weight=18, subsample=0.88, colsample_bytree=0.82,
            reg_alpha=0.15, reg_lambda=10.0, scale_pos_weight=scale_pw,
            max_bin=256, tree_method='hist', device=XGB_DEVICE,
            eval_metric='auc', early_stopping_rounds=450,
            random_state=s, n_jobs=-1, verbosity=0,
        )
        m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
        fold_val  += m.predict_proba(X_vl)[:, 1] / len(SEEDS_BASE)
        fold_test += m.predict_proba(X_te)[:, 1] / len(SEEDS_BASE)
        del m; gc.collect()

    oof_xgb[vl_idx] = fold_val
    test_xgb += fold_test / N_FOLDS
    a = roc_auc_score(y_vl, fold_val); fold_aucs_xgb.append(a)
    print(f"  fold {fold}: AUC = {a:.5f}")

xgb_auc = roc_auc_score(y, oof_xgb)
print(f"\n  XGBoost full-FE OOF AUC = {xgb_auc:.5f}  (mean fold {np.mean(fold_aucs_xgb):.5f})")

oof_preds['xgb_full_fe']  = oof_xgb
test_preds['xgb_full_fe'] = test_xgb
plot_model_diag('XGBoost full-FE', oof_xgb, y, COLORS['gold'])


  Training Model 2: XGBoost full-FE low-depth
  scale_pos_weight = 3.77
  fold 0: AUC = 0.95122
  fold 1: AUC = 0.95049
  fold 2: AUC = 0.95019
  fold 3: AUC = 0.95071
  fold 4: AUC = 0.95021

  XGBoost full-FE OOF AUC = 0.95056  (mean fold 0.95056)


np.float64(0.9505599446327306)

<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    Model 3 - LightGBM Full Feature View
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    A conservative LightGBM counterpart to XGBoost, trained on the same folds and feature set for leaf-wise diversity.
  </p>
</div>


In [31]:
print('=' * 70); print('  Training Model 3: LightGBM full-FE conservative'); print('=' * 70)

oof_lgb  = np.zeros(len(train_fe), dtype=np.float32)
test_lgb = np.zeros(len(test_fe),  dtype=np.float32)
fold_aucs_lgb = []

for fold, (tr_idx, vl_idx) in enumerate(folds_skf):
    X_tr = train_oe.iloc[tr_idx][FEATURE_COLS]
    X_vl = train_oe.iloc[vl_idx][FEATURE_COLS]
    X_te = test_oe[FEATURE_COLS]
    y_tr = y[tr_idx]; y_vl = y[vl_idx]

    fold_val  = np.zeros(len(vl_idx), dtype=np.float32)
    fold_test = np.zeros(len(test_fe), dtype=np.float32)

    for s in SEEDS_BASE:
        m = lgb.LGBMClassifier(
            n_estimators=9000, learning_rate=0.022, num_leaves=31,
            min_child_samples=120, subsample=0.88, subsample_freq=1,
            colsample_bytree=0.82, reg_alpha=0.25, reg_lambda=10.0,
            max_bin=255, class_weight='balanced',
            device=LGB_DEVICE, gpu_use_dp=False,
            random_state=s, n_jobs=-1, verbose=-1,
        )
        m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
              callbacks=[lgb.early_stopping(450, verbose=False), lgb.log_evaluation(0)])
        fold_val  += m.predict_proba(X_vl)[:, 1] / len(SEEDS_BASE)
        fold_test += m.predict_proba(X_te)[:, 1] / len(SEEDS_BASE)
        del m; gc.collect()

    oof_lgb[vl_idx] = fold_val
    test_lgb += fold_test / N_FOLDS
    a = roc_auc_score(y_vl, fold_val); fold_aucs_lgb.append(a)
    print(f"  fold {fold}: AUC = {a:.5f}")

lgb_auc = roc_auc_score(y, oof_lgb)
print(f"\n  LightGBM full-FE OOF AUC = {lgb_auc:.5f}  (mean fold {np.mean(fold_aucs_lgb):.5f})")

oof_preds['lgb_full_fe']  = oof_lgb
test_preds['lgb_full_fe'] = test_lgb
plot_model_diag('LightGBM full-FE', oof_lgb, y, COLORS['mid'])


  Training Model 3: LightGBM full-FE conservative


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  fold 0: AUC = 0.95968
  fold 1: AUC = 0.95906
  fold 2: AUC = 0.95878
  fold 3: AUC = 0.95948
  fold 4: AUC = 0.95928

  LightGBM full-FE OOF AUC = 0.95925  (mean fold 0.95925)


np.float64(0.9592513410211441)

<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    Model 4 - XGBoost on RealMLP Feature View
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    The same simple RealMLP-oriented feature view is encoded for XGBoost. This is the different-FE tree member in the stack.
  </p>
</div>


In [32]:
print('=' * 70); print('  Training Model 4: XGBoost on RealMLP-view FE'); print('=' * 70)

if RMLP_X is None:
    print("  SKIP XGB RealMLP-view because RealMLP feature view was not prepared.")
else:
    xgb_cols = RMLP_X.columns.tolist()
    rmlp_all = pd.concat([RMLP_X, RMLP_X_ORIG, RMLP_TEST], axis=0, ignore_index=True)
    cat_cols_rmlp = rmlp_all.select_dtypes(include=['object', 'category']).columns.tolist()

    rmlp_all_enc = rmlp_all.copy()
    for c in cat_cols_rmlp:
        codes, uniques = pd.factorize(rmlp_all_enc[c].astype(str), sort=False)
        rmlp_all_enc[c] = codes.astype(np.int32)
    rmlp_all_enc = rmlp_all_enc.replace([np.inf, -np.inf], np.nan).fillna(-999.0)

    X_rmlp_ps = rmlp_all_enc.iloc[:len(RMLP_X)].reset_index(drop=True)
    X_rmlp_orig = rmlp_all_enc.iloc[len(RMLP_X):len(RMLP_X)+len(RMLP_X_ORIG)].reset_index(drop=True)
    X_rmlp_test = rmlp_all_enc.iloc[len(RMLP_X)+len(RMLP_X_ORIG):].reset_index(drop=True)

    oof_xgb_rmlp  = np.zeros(len(train_fe), dtype=np.float32)
    test_xgb_rmlp = np.zeros(len(test_fe),  dtype=np.float32)
    fold_aucs_xgb_rmlp = []

    for fold, (tr_idx, vl_idx) in enumerate(folds_skf):
        X_tr = X_rmlp_ps.iloc[tr_idx].copy()
        y_tr = RMLP_Y.iloc[tr_idx].copy()
        if len(X_rmlp_orig) > 0:
            X_tr = pd.concat([X_tr, X_rmlp_orig], axis=0, ignore_index=True)
            y_tr = pd.concat([y_tr, RMLP_Y_ORIG], axis=0, ignore_index=True)

        X_vl = X_rmlp_ps.iloc[vl_idx]
        y_vl = RMLP_Y.iloc[vl_idx].values

        m = xgb.XGBClassifier(
            n_estimators=8000, learning_rate=0.025, max_depth=3,
            min_child_weight=16, subsample=0.90, colsample_bytree=0.86,
            reg_alpha=0.10, reg_lambda=9.0, scale_pos_weight=scale_pw,
            max_bin=256, tree_method='hist', device=XGB_DEVICE,
            eval_metric='auc', early_stopping_rounds=450,
            random_state=SEED + 777 + fold, n_jobs=-1, verbosity=0,
        )
        m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
        p_vl = m.predict_proba(X_vl)[:, 1]
        p_te = m.predict_proba(X_rmlp_test)[:, 1]

        oof_xgb_rmlp[vl_idx] = p_vl
        test_xgb_rmlp += p_te / N_FOLDS
        a = roc_auc_score(y_vl, p_vl); fold_aucs_xgb_rmlp.append(a)
        print(f"  fold {fold}: AUC = {a:.5f}")
        del m; gc.collect()

    xgb_rmlp_auc = roc_auc_score(y, oof_xgb_rmlp)
    print(f"\n  XGBoost RealMLP-view OOF AUC = {xgb_rmlp_auc:.5f}  (mean fold {np.mean(fold_aucs_xgb_rmlp):.5f})")
    oof_preds['xgb_realmlp_view'] = oof_xgb_rmlp
    test_preds['xgb_realmlp_view'] = test_xgb_rmlp
    plot_model_diag('XGBoost RealMLP-view', oof_xgb_rmlp, y, COLORS['secondary'])


  Training Model 4: XGBoost on RealMLP-view FE
  fold 0: AUC = 0.94828
  fold 1: AUC = 0.94732
  fold 2: AUC = 0.94724
  fold 3: AUC = 0.94761
  fold 4: AUC = 0.94742

  XGBoost RealMLP-view OOF AUC = 0.94757  (mean fold 0.94758)


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    Model 5 - Disabled
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    The old deep CatBoost/DART variants are intentionally removed from the active stack. A compact stack is less likely to overfit public noise.
  </p>
</div>


In [33]:
print('=' * 70)
print('  Model 5 disabled: compact RealMLP + GBDT stack selected')
print('=' * 70)


  Model 5 disabled: compact RealMLP + GBDT stack selected


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    Ensemble - OOF Search + LR Stacker + Committee
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li><b>Base pool:</b> RealMLP plus focused XGB/LGB models.</li>
  <li><b>OOF search:</b> raw and rank-normalized Dirichlet blends.</li>
  <li><b>Stacker:</b> regularized logistic regression on raw, rank, and logit meta-features.</li>
  <li><b>Final:</b> small committee over raw blend, rank blend, and LR stack.</li>
  </ul>
</div>


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📊 &nbsp; Step 1 - OOF Correlation Matrix
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Pairwise Pearson correlation between model OOF predictions. Pairs with correlation
    below 0.99 contribute meaningfully to ensemble lift; correlations above 0.997 add
    almost nothing. The Dart and Deep variants should show slightly lower correlation
    than the three base GBDTs.
  </p>
</div>


In [34]:
model_names = list(oof_preds.keys())
oof_mat  = np.stack([oof_preds[n]  for n in model_names], axis=1)
test_mat = np.stack([test_preds[n] for n in model_names], axis=1)

corr_oof = np.corrcoef(oof_mat.T)
fig = go.Figure(data=go.Heatmap(
    z=corr_oof, x=model_names, y=model_names,
    colorscale=DIVERGING, zmid=0.99, zmin=0.94, zmax=1.00,
    text=[[f"{v:.4f}" for v in row] for row in corr_oof],
    texttemplate='%{text}',
    textfont=dict(size=14, color='white', family='Arial Black'),
    colorbar=dict(title=dict(text='Pearson r', font=dict(color=COLORS['primary']))),
))
fig = style_fig(fig, 'OOF Correlation - Diversity Check', height=480)

avg_off_diag = (corr_oof.sum() - np.trace(corr_oof)) / (len(model_names) * (len(model_names) - 1))
add_in_graph_text(fig,
    f"<b>INSIGHT</b><br>"
    f"Average off-diagonal correlation: <b>{avg_off_diag:.4f}</b><br>"
    f"Lower-correlation models (Dart, Deep) get more weight automatically<br>"
    f"in the Dirichlet random search.",
    x=0.02, y=0.97, size=10)
fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🎲 &nbsp; Step 2 - Dirichlet Random Search (raw + rank)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Sample 3000 random weight vectors from a Dirichlet(α=1) distribution; pick the one
    that maximizes OOF AUC. Done in two modes: on raw probabilities and on rank-normalized
    probabilities. Random search is more conservative than greedy hill climbing - far less
    prone to overfitting OOF noise.
  </p>
</div>


In [35]:
def rank_norm(mat):
    """Rank-normalize each column to [0, 1]."""
    return np.argsort(np.argsort(mat, axis=0), axis=0) / max(len(mat) - 1, 1)

oof_rank  = rank_norm(oof_mat)
test_rank = rank_norm(test_mat)

def dirichlet_search(oof, y_true, n_cand=3000, alpha=1.0, seed=SEED):
    rng = np.random.RandomState(seed)
    n = oof.shape[1]
    best_auc, best_w = 0.0, None
    for _ in range(n_cand):
        w = rng.dirichlet(np.full(n, alpha))
        a = roc_auc_score(y_true, oof @ w)
        if a > best_auc:
            best_auc, best_w = a, w
    return best_w, best_auc

# RAW mode
w_raw, auc_raw = dirichlet_search(oof_mat, y, n_cand=3000)
blend_raw_oof  = oof_mat  @ w_raw
blend_raw_test = test_mat @ w_raw
print(f"  Dirichlet RAW  OOF AUC = {auc_raw:.5f}")
for n, w in zip(model_names, w_raw): print(f"      {n:18s}: {w:.4f}")

# RANK mode
w_rank, auc_rank = dirichlet_search(oof_rank, y, n_cand=3000, seed=SEED + 1)
blend_rank_oof  = oof_rank  @ w_rank
blend_rank_test = test_rank @ w_rank
print(f"\n  Dirichlet RANK OOF AUC = {auc_rank:.5f}")
for n, w in zip(model_names, w_rank): print(f"      {n:18s}: {w:.4f}")


  Dirichlet RAW  OOF AUC = 0.96153
      realmlp           : 0.4482
      xgb_full_fe       : 0.0076
      lgb_full_fe       : 0.5227
      xgb_realmlp_view  : 0.0216

  Dirichlet RANK OOF AUC = 0.96162
      realmlp           : 0.5090
      xgb_full_fe       : 0.0079
      lgb_full_fe       : 0.4458
      xgb_realmlp_view  : 0.0373


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📚 &nbsp; Step 3 - Logistic Regression Stacker (cross-fitted)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Meta-features: per-model raw + rank + logit transforms (3 transforms × 5 models = 15 features).
    Strong L2 regularization (C=0.2) prevents overfitting on OOF. Cross-fitted on the same fold
    splits used by the base models, so meta-features remain leak-free.
  </p>
</div>


In [36]:
def make_meta(mat):
    rk = np.argsort(np.argsort(mat, axis=0), axis=0) / max(len(mat) - 1, 1)
    lg = logit(mat.clip(1e-4, 1 - 1e-4))
    return np.hstack([mat, rk, lg]).astype(np.float32)

meta_train = make_meta(oof_mat)
meta_test  = make_meta(test_mat)

stack_oof  = np.zeros(len(y),       dtype=np.float32)
stack_test = np.zeros(len(test_fe), dtype=np.float32)

for fold, (tr_idx, vl_idx) in enumerate(folds_skf):
    sc = StandardScaler()
    Xs_tr = sc.fit_transform(meta_train[tr_idx])
    Xs_vl = sc.transform(meta_train[vl_idx])
    Xs_te = sc.transform(meta_test)
    m = LogisticRegression(C=0.2, max_iter=1500, solver='lbfgs', class_weight='balanced')
    m.fit(Xs_tr, y[tr_idx])
    stack_oof[vl_idx] = m.predict_proba(Xs_vl)[:, 1]
    stack_test += m.predict_proba(Xs_te)[:, 1] / N_FOLDS

stack_lr_auc = roc_auc_score(y, stack_oof)
print(f"  LR-Stacker OOF AUC = {stack_lr_auc:.5f}")


  LR-Stacker OOF AUC = 0.96234


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🏛️ &nbsp; Step 4 - Final Committee Blend
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Three ensemble candidates: Dirichlet-raw, Dirichlet-rank, LR-stack. Final weights
    are found by another small Dirichlet random search (4000 candidates, α=1.5 for
    a slightly more uniform prior). This mirrors the L3 committee structure of the
    public 0.9513 baseline.
  </p>
</div>


In [37]:
committee_oof  = np.stack([blend_raw_oof,  blend_rank_oof,  stack_oof],  axis=1)
committee_test = np.stack([blend_raw_test, blend_rank_test, stack_test], axis=1)
committee_names = ['blend_raw', 'blend_rank', 'lr_stack']

cm_w, cm_auc = dirichlet_search(committee_oof, y, n_cand=4000, alpha=1.5, seed=SEED + 2)
final_oof  = committee_oof  @ cm_w
final_test = committee_test @ cm_w

print('=' * 70)
print(f"  FINAL COMMITTEE OOF AUC = {cm_auc:.5f}")
print('=' * 70)
for n, w in zip(committee_names, cm_w): print(f"      {n:18s}: {w:.4f}")

base_max = max(roc_auc_score(y, oof_preds[n]) for n in model_names)
print(f"\n  Best base model AUC: {base_max:.5f}")
print(f"  Ensemble lift      : {cm_auc - base_max:+.5f}")


  FINAL COMMITTEE OOF AUC = 0.96234
      blend_raw         : 0.0323
      blend_rank        : 0.0369
      lr_stack          : 0.9308

  Best base model AUC: 0.95926
  Ensemble lift      : +0.00307


<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    🎨 &nbsp; Ensemble Dashboard (4-panel)
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
    Final confusion matrix, ROC overlay across all models, OOF AUC ranking, and the
    final committee weights - all in a single dashboard.
  </p>
</div>


In [38]:
all_aucs = {n: roc_auc_score(y, oof_preds[n]) for n in model_names}
all_aucs['blend_raw']  = auc_raw
all_aucs['blend_rank'] = auc_rank
all_aucs['lr_stack']   = stack_lr_auc
all_aucs['FINAL']      = cm_auc

final_label = (final_oof >= 0.5).astype(int)
cm_final = confusion_matrix(y, final_label)
fpr_f, tpr_f, _ = roc_curve(y, final_oof)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Final Confusion Matrix', 'ROC Curves - All Models',
                    'OOF AUC Ranking', 'Committee Weights'),
    specs=[[{'type':'heatmap'}, {'type':'xy'}],
           [{'type':'xy'},      {'type':'domain'}]],
    column_widths=[0.42, 0.58], row_heights=[0.50, 0.50],
)

fig.add_trace(go.Heatmap(
    z=cm_final, x=['Pred 0','Pred 1'], y=['Actual 0','Actual 1'],
    colorscale=[[0, COLORS['bg']], [1, COLORS['accent']]],
    text=cm_final, texttemplate='%{text}', showscale=False,
    textfont=dict(size=22, color=COLORS['primary'], family='Arial Black')),
    row=1, col=1)

roc_palette = PALETTE_EXT[:len(model_names) + 1]
for i, n in enumerate(model_names):
    fpr_, tpr_, _ = roc_curve(y, oof_preds[n])
    fig.add_trace(go.Scatter(x=fpr_, y=tpr_, mode='lines', name=n,
                             line=dict(color=roc_palette[i], width=1.6, dash='dot')), row=1, col=2)
fig.add_trace(go.Scatter(x=fpr_f, y=tpr_f, mode='lines', name='FINAL',
                         line=dict(color=COLORS['gold'], width=4)), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', showlegend=False,
                         line=dict(color=COLORS['soft'], dash='dash', width=1)), row=1, col=2)

sorted_aucs = sorted(all_aucs.items(), key=lambda x: x[1])
labels_ = [k for k, _ in sorted_aucs]
vals_   = [v for _, v in sorted_aucs]
bar_colors = [COLORS['gold'] if k == 'FINAL' else
              (COLORS['accent'] if k in ['blend_raw', 'blend_rank', 'lr_stack'] else COLORS['mid'])
              for k in labels_]
fig.add_trace(go.Bar(
    y=labels_, x=vals_, orientation='h',
    marker=dict(color=bar_colors, line=dict(color=COLORS['primary'], width=1.2)),
    text=[f"{v:.5f}" for v in vals_], textposition='outside',
    textfont=dict(size=11, color=COLORS['primary'], family='Arial Black'),
    showlegend=False), row=2, col=1)

fig.add_trace(go.Pie(
    labels=committee_names, values=cm_w, hole=0.50,
    marker=dict(colors=[COLORS['mid'], COLORS['secondary'], COLORS['accent']],
                line=dict(color=COLORS['primary'], width=2)),
    textinfo='label+percent',
    textfont=dict(size=12, color='white', family='Arial Black')), row=2, col=2)

fig = style_fig(fig, f'ENSEMBLE DASHBOARD - Final OOF AUC = {cm_auc:.5f}', height=850)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=2)
fig.update_yaxes(title_text='True Positive Rate',  row=1, col=2)
fig.update_xaxes(title_text='OOF AUC', row=2, col=1, range=[min(vals_) - 0.001, max(vals_) + 0.001])
add_in_graph_text(fig,
    f"<b>FINAL RESULT</b><br>OOF AUC = <b>{cm_auc:.5f}</b><br>"
    f"Best base = {max(roc_auc_score(y, oof_preds[n]) for n in model_names):.5f}",
    x=0.02, y=0.97, size=11)
fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 55%, #0F3460 100%);
     padding: 28px 32px; border-radius: 14px; border-left: 8px solid #E94560;
     box-shadow: 0 8px 24px rgba(26,26,46,0.28); margin-top: 25px;">
  <h1 style="color: #F5A623; margin: 0; font-size: 30px; font-weight: bold;
             letter-spacing: 1px; font-family: Georgia, serif;">
    💾 &nbsp; Submission Generation
  </h1>
  <ul style="color: #F8F9FC; font-size: 15px; line-height: 1.85; margin: 14px 0 0 0;">
  <li>Predictions clip to [0,1] for safety</li>
  <li>Format: <code>id, PitNextLap</code> (probabilistic, AUC scoring)</li>
  <li>Output file: <code>submission.csv</code> in working directory</li>
  </ul>
</div>


In [39]:
def _clip01(a):
    return np.clip(np.asarray(a, dtype=np.float64), 0.0, 1.0)


def _write_submission(name, pred):
    sub = pd.DataFrame({
        ID_COL: test_fe[ID_COL].values,
        TARGET: _clip01(pred),
    })
    sub.to_csv(name, index=False)
    print(f"  wrote {name:36s} | mean={sub[TARGET].mean():.6f} std={sub[TARGET].std():.6f}")
    return sub


# Public LB was slightly worse with the aggressive LR committee, so default to
# a conservative high-OOF two-model blend. The final committee is still saved.
realmlp_test = test_preds.get('realmlp')
lgb_test = test_preds.get('lgb_full_fe')
xgb_test = test_preds.get('xgb_full_fe')
xgb_rmlp_test = test_preds.get('xgb_realmlp_view')

safe_candidates = {}
if realmlp_test is not None:
    safe_candidates['realmlp_single'] = realmlp_test
if lgb_test is not None:
    safe_candidates['lgb_single'] = lgb_test
if realmlp_test is not None and lgb_test is not None:
    safe_candidates['safe_52_realmlp_48_lgb'] = 0.52 * realmlp_test + 0.48 * lgb_test
    safe_candidates['safe_60_realmlp_40_lgb'] = 0.60 * realmlp_test + 0.40 * lgb_test
    safe_candidates['safe_45_realmlp_55_lgb'] = 0.45 * realmlp_test + 0.55 * lgb_test
if realmlp_test is not None and lgb_test is not None and xgb_test is not None and xgb_rmlp_test is not None:
    safe_candidates['safe_oof_raw_like'] = (
        0.45 * realmlp_test + 0.52 * lgb_test + 0.01 * xgb_test + 0.02 * xgb_rmlp_test
    )

safe_candidates['blend_raw_oof_search'] = blend_raw_test
safe_candidates['blend_rank_oof_search'] = blend_rank_test
safe_candidates['final_lr_committee'] = final_test

print('=' * 70)
print('  Writing submission variants')
print('=' * 70)

for key, pred in safe_candidates.items():
    _write_submission(f"submission_{key}.csv", pred)

default_key = 'safe_52_realmlp_48_lgb' if 'safe_52_realmlp_48_lgb' in safe_candidates else 'realmlp_single'
submission = _write_submission('submission.csv', safe_candidates[default_key])

print(f"\n✅  default submission.csv = {default_key}")
print(submission.head())
print('\n  Default prediction distribution:')
print(submission[TARGET].describe().to_string())


  Writing submission variants
  wrote submission_realmlp_single.csv        | mean=0.197843 std=0.309807
  wrote submission_lgb_single.csv            | mean=0.274378 std=0.361395
  wrote submission_safe_52_realmlp_48_lgb.csv | mean=0.234580 std=0.330306
  wrote submission_safe_60_realmlp_40_lgb.csv | mean=0.228457 std=0.326292
  wrote submission_safe_45_realmlp_55_lgb.csv | mean=0.239937 std=0.333999
  wrote submission_safe_oof_raw_like.csv     | mean=0.240576 std=0.333775
  wrote submission_blend_raw_oof_search.csv  | mean=0.240704 std=0.333872
  wrote submission_blend_rank_oof_search.csv | mean=0.500000 std=0.285858
  wrote submission_final_lr_committee.csv    | mean=0.296086 std=0.363511
  wrote submission.csv                       | mean=0.234580 std=0.330306

✅  default submission.csv = safe_52_realmlp_48_lgb
       id  PitNextLap
0  439140    0.006770
1  439141    0.021678
2  439142    0.010313
3  439143    0.309199
4  439144    0.876264

  Default prediction distribution:
count  

<div style="background: linear-gradient(90deg, #F8F9FC 0%, #FCE9EE 100%);
     padding: 18px 22px; border-radius: 10px; border-left: 5px solid #E94560;
     border-top: 1px solid #E94560; margin-top: 18px;">
  <h2 style="color: #1A1A2E; margin: 0 0 6px 0; font-size: 22px; font-family: Georgia, serif;">
    📋 &nbsp; Final Summary Table
  </h2>
  <p style="color: #16213E; font-size: 14px; line-height: 1.7; margin: 0;">
   All models, ensemble layers, and the final score in a single visual.
  </p>
</div>


In [40]:
summary_rows = []
for n in model_names:
    summary_rows.append([n, roc_auc_score(y, oof_preds[n]), 'base model'])
summary_rows += [
    ['Dirichlet_raw',   auc_raw,       'L2 ensemble'],
    ['Dirichlet_rank',  auc_rank,      'L2 ensemble'],
    ['LR_stack',        stack_lr_auc,  'L2 stacker'],
    ['FINAL committee', cm_auc,        'L3 final'],
]

fig = go.Figure(data=[go.Table(
    header=dict(values=['<b>Model / Stage</b>', '<b>OOF AUC</b>', '<b>Tier</b>'],
                fill_color=COLORS['primary'],
                font=dict(color=COLORS['gold'], size=14, family='Georgia, serif'),
                align='left', height=40),
    cells=dict(values=[[r[0] for r in summary_rows],
                       [f"{r[1]:.5f}" for r in summary_rows],
                       [r[2] for r in summary_rows]],
               fill_color=[['white' if r[2] != 'L3 final' else COLORS['highlight'] for r in summary_rows]]*3,
               font=dict(color=COLORS['primary'], size=13, family='Georgia, serif'),
               align='left', height=32),
)])
fig = style_fig(fig, 'Model Performance Summary', height=520)
fig.show(renderer='iframe_connected')


<div style="background: linear-gradient(135deg, #1A1A2E 0%, #16213E 50%, #0F3460 100%);
     padding: 45px 30px; border-radius: 22px; text-align: center;
     box-shadow: 0 14px 40px rgba(26,26,46,0.45);
     border: 2px solid #F5A623; margin-top: 30px;">

  <h1 style="color: #F5A623; margin:0; font-size: 38px; letter-spacing: 2px; font-family: Georgia, serif;">
    🏁 &nbsp; THANK YOU FOR READING &nbsp; 🏆
  </h1>
</div>
